# 模型输入：20 日滚动分位数

从原始 Alpha 中选择固定顺序的 45 列。每列独立执行：**逐日横截面分位数 → 最近 20 日等权平均 → 再次横截面分位数**，生成尺度统一的模型输入。

日期 T 的输入仅使用 T−20 至 T−1 的因子，输出到 `rank_output/factor_rank_20日/年份/日期.parquet`。

In [ ]:
import pandas as pd
from collections import deque
from tqdm import tqdm

from TOOL.data import preload, date_path, save_scores, OUTPUT_PATH, RANK_PATH
from TOOL.Struct import MarketField
from TOOL.factor import ALPHA_LIST, averRank

In [ ]:
WINDOW = 20
CACHE_SIZE = 30
years_list = ['2020', '2021', '2022', '2023', '2024', '2025', '2026']
base_info, dates_list, dates_path_list = preload(years_list)
factors_path_list = [date_path(OUTPUT_PATH, date) for date in dates_list]
factor_cache = deque(maxlen=CACHE_SIZE)

## 生成当日输入，再更新缓存

`DAY >= WINDOW + 1` 时历史窗口完整，可以生成当天的 `rank_today`。计算结束后才将当天原始因子加入 `factor_cache`，使当天开盘使用的信号不包含当天收盘信息。缓存长度为 30 日，实际均值窗口为 20 日，跨年连续推进。

In [ ]:
DAY = 0
for date, factor_path in tqdm(list(zip(dates_list, factors_path_list)), mininterval=30):
    DAY += 1
    factor_info = pd.read_parquet(factor_path, columns=['order_book_id', 'datetime', *ALPHA_LIST])

    #=======
    if DAY >= WINDOW + 1:
        factor_summary = MarketField(pd.concat(factor_cache))
        rank_today = pd.DataFrame(index=factor_summary.assets)
        for alpha in ALPHA_LIST:
            rank_today[alpha] = averRank(factor_summary, alpha, WINDOW)
        save_scores(rank_today, RANK_PATH, date)

    factor_cache.append(factor_info)
    #=======

In [ ]:
print('保存目录：', RANK_PATH)
print('文件数：', len(list(RANK_PATH.glob('*/*.parquet'))))
rank_today.iloc[:5, :8]